# 02. Noticias y contexto para los gaps

En este notebook reunimos las noticias y los comunicados publicados **antes de la apertura** de cada gap. Ocultamos el nombre de la empresa, el ticker y las fechas para que el modelo de lenguaje valore únicamente los documentos, sin conocer el resultado posterior.

El LLM asignará una puntuación de **0 a 3** según la importancia de las noticias. Después comprobaremos si esta información mejora las predicciones basadas en datos de mercado.

Los documentos deben haberse descargado previamente en `data/noticias`, `data/filings_eventos.csv` y `data/filings_texto`. Para probar el proceso con pocos eventos se puede cambiar `NUM_CASOS` a 5.

## 1. Leer los eventos y los documentos

In [ ]:
from pathlib import Path
import json
import re
import pandas as pd

from dotenv import load_dotenv

load_dotenv()

DATOS = Path("data")
NUM_CASOS = None  # Pon 5 para probar con pocos gaps.

eventos = pd.read_csv(DATOS / "eventos_gap40.csv", parse_dates=["fecha"])
empresas = pd.read_csv(DATOS / "universo.csv", keep_default_na=False)
eventos = eventos.merge(empresas[["symbol", "name"]], on="symbol", how="left")
# Nombres anteriores de cada empresa según la SEC, para ocultarlos también.
nombres_sec = json.loads((DATOS / "nombres_cik.json").read_text(encoding="utf-8"))

filings = pd.read_csv(DATOS / "filings_eventos.csv", keep_default_na=False)
filings["horas_antes"] = pd.to_numeric(filings["horas_antes"])

if NUM_CASOS is not None:
    eventos = eventos.head(NUM_CASOS).copy()
print("Gaps seleccionados:", len(eventos))

Gaps seleccionados: 1818


## 2. Preparar los textos

Para cada evento seleccionamos las noticias de los **30 días anteriores** y los comunicados de la SEC de los **90 días anteriores**. Ordenamos los documentos desde el más reciente, limitamos su extensión y ocultamos los datos que podrían identificar a la empresa.

No incluimos información sobre cómo terminó la sesión, porque el LLM solo debe utilizar lo que se sabía antes de la apertura.

In [ ]:
from bs4 import BeautifulSoup

textos = []
for evento in eventos.itertuples():
    codigo = str(evento.id_evento)  # Identificador estable aunque cambie el orden de los eventos
    apertura = pd.Timestamp(evento.fecha).tz_localize("America/New_York") + pd.Timedelta(hours=9, minutes=30)
    documentos = []

    # Noticias ya descargadas.
    archivo_noticias = DATOS / "noticias" / f"{evento.id_evento}.json"
    noticias = json.loads(archivo_noticias.read_text(encoding="utf-8")) if archivo_noticias.exists() else []
    for noticia in noticias:
        publicada = pd.Timestamp(noticia["publicado"]).tz_convert("America/New_York")
        horas = (apertura - publicada).total_seconds() / 3600
        if 0 < horas <= 30 * 24:
            contenido = noticia.get("contenido") or noticia.get("resumen") or ""
            titular = noticia.get("titular") or ""
            texto = titular + ". " + BeautifulSoup(contenido, "html.parser").get_text(" ")
            documentos.append((horas, "Noticia", texto[:8000]))

    # Comunicados SEC ya descargados.
    comunicados = filings[filings["id_evento"] == evento.id_evento]
    for comunicado in comunicados.itertuples():
        if 0 < comunicado.horas_antes <= 90 * 24:
            archivo = DATOS / "filings_texto" / f"{comunicado.accessionNumber}.txt"
            contenido = archivo.read_text(encoding="utf-8") if archivo.exists() else ""
            limite = 8000 if comunicado.form.startswith("8-K") else 4000  # de los registros de emisión basta el principio
            texto = f"Comunicado SEC {comunicado.form}. {contenido[:limite]}"
            documentos.append((comunicado.horas_antes, "SEC", texto))

    documentos.sort(key=lambda x: x[0])
    documentos = documentos[:40]

    # Eliminamos nombres, ticker y fechas identificables antes de enviar los textos.
    nombres = [str(evento.name or "").strip()]
    nombres += [nombres_sec.get(str(evento.cik), {}).get("nombre", "")] + nombres_sec.get(str(evento.cik), {}).get("anteriores", [])
    partes = ["Documentos anteriores a la apertura:"]
    total = 0
    for horas, tipo, texto in documentos:
        for nombre in nombres:
            if nombre:
                texto = re.sub(re.escape(nombre), "[EMPRESA]", texto, flags=re.IGNORECASE)
        texto = re.sub(r"(?<![A-Za-z0-9])" + re.escape(evento.symbol) + r"(?![A-Za-z0-9])", "[TICKER]", texto)
        texto = re.sub(r"\b(?:19|20)\d{2}-\d{2}-\d{2}\b", "[FECHA]", texto)
        texto = re.sub(r"\b(?:19|20)\d{2}\b", "[AÑO]", texto)
        if total + len(texto) > 60000:
            texto = texto[:max(0, 60000 - total)]
        total += len(texto)
        partes.append(f"{tipo}, {horas:.0f} horas antes:\n{texto}")

    textos.append({"id_evento": evento.id_evento, "codigo": codigo,
                   "documentos": len(documentos), "texto": "\n\n".join(partes)})

casos = pd.DataFrame(textos)
print(casos[["codigo", "documentos"]].head().to_string(index=False))
print("\nEJEMPLO:\n", casos.iloc[0]["texto"][:1200] if len(casos) else "Sin eventos")

C:\Users\mmacias\AppData\Local\Temp\ipykernel_37424\2322182316.py:18: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  texto = titular + ". " + BeautifulSoup(contenido, "html.parser").get_text(" ")


         codigo  documentos
 SLS_2021-02-03          11
AAME_2021-02-05           4
AREC_2021-02-05          24
CHCI_2021-02-09           4
 OLB_2021-02-10           5

EJEMPLO:
 Documentos anteriores a la apertura:

Noticia, 1 horas antes:
12 Health Care Stocks Moving In Wednesday's Pre-Market Session.   
 Gainers 
 
 SELLAS Life Sciences Gr  (NASDAQ: [TICKER] ) shares rose 60.14% to $13.1 during Wednesday's pre-market session. The company's market cap stands at $185.9 million. 
 Tyme Technologies  (NASDAQ: TYME ) shares increased by 57.69% to $2.87. The company's market cap stands at $373.5 million. 
 GW Pharmaceuticals  (NASDAQ: GWPH ) shares rose 47.0% to $215.0. The company's market cap stands at $6.7 billion. 
 Selecta Biosciences  (NASDAQ: SELB ) stock increased by 19.63% to $5.3. The company's market cap stands at $572.5 million. 
 Zynerba Pharmaceuticals  (NASDAQ: ZYNE ) shares increased by 18.82% to $4.23. The market value of their outstanding shares is at $124.5 million. 
 S

**Comprobación:** antes de enviar los textos al LLM, conviene leer varios ejemplos para verificar que no aparecen nombres de empresas, tickers ni fechas. Estas sustituciones son aproximadas y pueden dejar referencias identificables.

## 3. Qué le pedimos al LLM

El LLM **no intenta predecir el precio**. Solo puntúa si los documentos contienen una noticia que podría explicar una subida importante en la apertura. Usamos la misma escala para todos los casos y guardamos una explicación breve de cada puntuación.

In [ ]:
PROMPT = """Analiza las noticias y los comunicados publicados antes de la apertura de una acción.

Valora si contienen algún hecho concreto que pueda explicar una subida importante.
Usa únicamente los documentos proporcionados. No intentes identificar la empresa,
no utilices información posterior y no predigas lo que hará su precio.

Asigna una puntuación:
0 = No hay ninguna noticia que explique la subida.
1 = Hay una noticia, pero su importancia es limitada o poco clara.
2 = Hay una noticia relevante que podría explicar la subida.
3 = Hay un acontecimiento muy importante y concreto, como una aprobación
    regulatoria definitiva, una adquisición confirmada o resultados extraordinarios.

Una noticia que sólo dice que la acción está subiendo no explica por qué sube.
Devuelve la puntuación y una explicación breve en el formato solicitado.
"""

# Solo pedimos los dos datos que utilizaremos en el análisis.
ESQUEMA = {
    "type": "object",
    "properties": {
        "solidez": {"type": "integer", "enum": [0, 1, 2, 3]},
        "explicacion": {"type": "string"}
    },
    "required": ["solidez", "explicacion"],
    "additionalProperties": False
}

print(PROMPT)

Analiza las noticias y los comunicados publicados antes de la apertura de una acción.

Valora si contienen algún hecho concreto que pueda explicar una subida importante.
Usa únicamente los documentos proporcionados. No intentes identificar la empresa,
no utilices información posterior y no predigas lo que hará su precio.

Asigna una puntuación:
0 = No hay ninguna noticia que explique la subida.
1 = Hay una noticia, pero su importancia es limitada o poco clara.
2 = Hay una noticia relevante que podría explicar la subida.
3 = Hay un acontecimiento muy importante y concreto, como una aprobación
    regulatoria definitiva, una adquisición confirmada o resultados extraordinarios.

Una noticia que solo dice que la acción está subiendo no explica por qué sube.
Devuelve la puntuación y una explicación breve en el formato solicitado.



## 4. Consultar al LLM

Las respuestas se guardan en `data/senales_llm_simple`, separadas de las obtenidas con el prompt anterior. Así no mezclamos puntuaciones generadas con criterios distintos.

No enviamos casos sin documentos: se registran por separado en el CSV final. Antes de activar las llamadas de pago, revisa una muestra de los textos y comprueba que tienes configurada la clave `ANTHROPIC_API_KEY`.

In [ ]:
ENVIAR = False
MODELO = "claude-haiku-5-5"
carpeta_respuestas = DATOS / "senales_llm_simple"

if ENVIAR:
    import anthropic
    cliente = anthropic.Anthropic()
    carpeta_respuestas.mkdir(exist_ok=True)

    for caso in casos.itertuples():
        if caso.documentos == 0:
            continue

        salida = carpeta_respuestas / f"{caso.codigo}.json"
        if salida.exists():
            continue

        respuesta = cliente.messages.create(
            model=MODELO,
            max_tokens=1000,
            thinking={"type": "disabled"},
            system=PROMPT,
            messages=[{"role": "user", "content": caso.texto}],
            output_config={
                "format": {
                    "type": "json_schema",
                    "schema": ESQUEMA
                }
            }
        )

        if respuesta.stop_reason != "end_turn":
            raise ValueError(f"Respuesta incompleta: {respuesta.stop_reason}")

        texto = next(
            bloque.text for bloque in respuesta.content
            if bloque.type == "text"
        )

        valoracion = json.loads(texto)

        # Guardamos cada valoración antes de pasar al siguiente evento
        salida.write_text(
            json.dumps(
                {"id_evento": caso.id_evento, **valoracion},
                ensure_ascii=False,
                indent=2
            ),
            encoding="utf-8"
        )

    print("Consultas terminadas.")
else:
    print("No se han realizado consultas de pago.")

Consultas terminadas.


## 5. Guardar el contexto para los modelos

La puntuación 0 significa que **sí había documentos**, pero ninguno contenía una noticia relevante. Cuando **no hay documentos**, guardamos `sin_documentos = 1` para distinguir ambas situaciones.

Si un caso tiene documentos pero todavía no ha sido evaluado, su puntuación queda vacía. No debe tratarse como un 0.

In [ ]:
# Reunimos todos los eventos, aunque todavía falten respuestas del LLM.
contexto = casos[["id_evento", "documentos"]].copy()
contexto["sin_documentos"] = (contexto["documentos"] == 0).astype(int)
contexto["solidez"] = pd.NA
contexto["explicacion"] = ""

for i, caso in contexto.iterrows():
    if caso["sin_documentos"] == 1:
        continue
    archivo = carpeta_respuestas / f"{caso['id_evento']}.json"
    if archivo.exists():
        respuesta = json.loads(archivo.read_text(encoding="utf-8"))
        contexto.at[i, "solidez"] = respuesta["solidez"]
        contexto.at[i, "explicacion"] = respuesta["explicacion"]

contexto["solidez"] = pd.to_numeric(contexto["solidez"], errors="coerce")
pendientes = ((contexto["sin_documentos"] == 0) & contexto["solidez"].isna()).sum()

print("Eventos sin documentos:", contexto["sin_documentos"].sum())
print("Eventos pendientes de valorar:", pendientes)

if pendientes == 0:
    # El 0 de estos casos significa ausencia de documentos, no una valoración del LLM.
    contexto.loc[contexto["sin_documentos"] == 1, "solidez"] = 0
    destino = DATOS / ("contexto_gap40.csv" if NUM_CASOS is None else "contexto_gap40_prueba.csv")
    contexto.to_csv(destino, index=False)
    print("Archivo guardado:", destino)
else:
    print("No se guarda el CSV final hasta que todos los casos con documentos estén valorados.")

Eventos sin documentos: 16
Eventos pendientes de valorar: 0
Archivo guardado: data\contexto_gap40.csv


### Limitaciones

El modelo puede equivocarse al valorar las noticias y la ocultación de nombres y fechas no es perfecta. Además, los textos se recortan para limitar el tamaño de las consultas. Por eso conviene revisar manualmente una muestra de documentos y puntuaciones.

**Importante:** este prompt es distinto del anterior. Para utilizarlo en el TFM hay que generar nuevas puntuaciones y actualizar los resultados de los modelos; no deben presentarse las respuestas antiguas como si se hubieran obtenido con estas instrucciones.